In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'


## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [2]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['categories_l1'] = category_list['categories_l1'].str.lower()
category_list['categories_l2'] = category_list['categories_l2'].str.lower()
category_list = category_list[['app_name', 'categories_l1', 'categories_l2']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app']
['educational' 'rest' 'office' 'ride haling' 'gaming' 'finance_investment'
 'driver_app']


,app_name,categories_l1,categories_l2
0,ignou e-content,educational,educational


In [3]:
##Parameter 
start_date = '20240322'
end_date = '20240428'

data = f"""

    WITH rewards AS (
    
    SELECT
        event_props_user_id user_id,
        MAX(CASE WHEN event_props_action = 'ctaClicked' THEN event_props_action END) AS cta_clicked,
        MAX(CASE WHEN event_props_action = 'copyCodeClicked' THEN event_props_action END) AS copy_code_clicked
        
    FROM 
        clevertap.customer_rewards_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        and event_props_reward_partner in ('My11 Circle', 'My 11Circle')
        
    GROUP BY 1
    ),
    
    customer_app AS (
    
    SELECT
        rewards.*,
        date_format(from_unixtime(updated_epoch/1000), '%Y%m%d') AS last_sync_date,
        id, 
        json_extract(data, '$.appographyData') as app_lists,
        DENSE_RANK() over(partition by id order by yyyymmdd DESC) updated_seq
    FROM 
        rewards 
    LEFT JOIN 
        raw_internal.kafka_info_customer_installed_apps_v1
        ON rewards.user_id = id
    WHERE 
        yyyymmdd >= DATE_FORMAT((DATE_PARSE('{start_date}', '%Y%m%d') - INTERVAL '30' DAY), '%Y%m%d')
        AND yyyymmdd <= DATE_FORMAT((DATE_PARSE('{end_date}', '%Y%m%d') + INTERVAL '15' DAY), '%Y%m%d')
        AND id IN (SELECT user_id FROM rewards)
    )
    
    SELECT 
        user_id,
        cta_clicked,
        copy_code_clicked,
        id,
        RTRIM(LTRIM(LOWER(app_list))) app_list
    FROM 
        (
        SELECT
            user_id,
            cta_clicked,
            copy_code_clicked,
            id,
            last_sync_date,
            app_lists
        FROM 
            customer_app
        WHERE 
            updated_seq = 1
        ) t
    CROSS JOIN UNNEST(cast(app_lists as array(varchar))) AS t(app_list)

    GROUP BY 1,2,3,4,5


"""
df_data = pd.read_sql(data, connection)

In [4]:
df_data['app_list'] = df_data['app_list'].str.strip()
df_data.drop_duplicates(inplace=True)

df_merge = pd.merge(df_data, 
                    category_list,
                    how='left',
                    left_on='app_list',
                    right_on='app_name'
                   )
df_merge['values'] = 1
df_merge

,user_id,cta_clicked,copy_code_clicked,id,app_list,app_name,categories_l1,categories_l2,values
0,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,amazon prime video,amazon prime video,ott,rest,1
1,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
2,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
3,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,alt balaji,alt balaji,ott,rest,1
4,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,bookmyshow,bookmyshow,entertainment,rest,1
...,...,...,...,...,...,...,...,...,...
2331347,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola,ola,travel_ridehailing,ride haling,1
2331348,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola driver app,ola driver app,driver_app,driver_app,1
2331349,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,rapido captain,rapido captain,driver_app,driver_app,1
2331350,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,snapchat,snapchat,social,rest,1


### Apps

In [5]:
df1 = df_merge.groupby(['app_list'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df1['total_customers'] = df_merge.user_id.nunique()
df1['%'] = (df1['count']*100.00/df1['total_customers']).round(2) 

In [7]:
df1.sort_values(by='%', ascending=False).head(60)

,app_list,count,total_customers,%
0,google maps,100722,100770,99.95
1,youtube,100474,100770,99.71
2,gmail,100429,100770,99.66
3,whatsapp,97229,100770,96.49
4,google photos,93904,100770,93.19
5,instagram,81833,100770,81.21
6,facebook,73284,100770,72.72
7,truecaller,67906,100770,67.39
8,google calendar,61272,100770,60.80
9,flipkart,58242,100770,57.80


### Categories

In [9]:
df2 = df_merge.groupby(['categories_l1'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df2['total_customers'] = df_merge.user_id.nunique()
df2['%'] = (df2['count']*100.00/df2['total_customers']).round(2) 

In [10]:
df2.sort_values(by='%', ascending=False).head(60)

,categories_l1,count,total_customers,%
0,tools,100765,100770,100.00
1,social,100592,100770,99.82
2,ott,78234,100770,77.64
3,ecommerce,72496,100770,71.94
4,finance_transactions,64575,100770,64.08
5,travel_ridehailing,61721,100770,61.25
6,streaming_music,60862,100770,60.40
7,delivery_food,54651,100770,54.23
8,office,45440,100770,45.09
9,news,31105,100770,30.87


### Remove most common apps

In [53]:
most_common_apps = ['google maps', 'youtube', 'gmail', 'whatsapp', 'google photos', 
                    'google calendar','clock', 'google news', 'hangouts', 'calculator',
                    'accuweather', 'truecaller'
                   ]

In [35]:
df3 = df_merge[~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['app_list'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df3['total_customers'] = df_merge.user_id.nunique()
df3['%'] = (df3['count']*100.00/df3['total_customers']).round(2) 

In [36]:
df3.head(40)

,app_list,count,total_customers,%
0,instagram,81833,100770,81.21
1,facebook,73284,100770,72.72
2,truecaller,67906,100770,67.39
3,flipkart,58242,100770,57.80
4,paytm money,58140,100770,57.70
5,jio cinema,52198,100770,51.80
6,snapchat,51236,100770,50.84
7,telegram,48416,100770,48.05
8,disney+ hotstar,46718,100770,46.36
9,spotify,46294,100770,45.94


In [37]:
df4 = df_merge[~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['categories_l1'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df4['total_customers'] = df_merge.user_id.nunique()
df4['%'] = (df4['count']*100.00/df4['total_customers']).round(2) 

In [38]:
df4.sort_values(by='%', ascending=False).head(60)

,categories_l1,count,total_customers,%
0,social,98018,100770,97.27
1,tools,79281,100770,78.68
2,ott,78234,100770,77.64
3,ecommerce,72496,100770,71.94
4,finance_transactions,64575,100770,64.08
5,travel_ridehailing,61721,100770,61.25
6,streaming_music,60862,100770,60.40
7,delivery_food,54651,100770,54.23
8,office,45440,100770,45.09
9,finance_investment,28357,100770,28.14


### Check

In [39]:
df5 = df_merge[~df_merge['copy_code_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['app_list'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df5['total_customers'] = df_merge.user_id.nunique()
df5['%'] = (df5['count']*100.00/df5['total_customers']).round(2) 

In [40]:
df5.head(40)

,app_list,count,total_customers,%
0,instagram,81763,100770,81.14
1,facebook,73220,100770,72.66
2,truecaller,67852,100770,67.33
3,flipkart,58194,100770,57.75
4,paytm money,58091,100770,57.65
5,jio cinema,52160,100770,51.76
6,snapchat,51190,100770,50.80
7,telegram,48381,100770,48.01
8,disney+ hotstar,46677,100770,46.32
9,spotify,46253,100770,45.90


In [41]:
df_merge[~df_merge['copy_code_clicked'].isna()]

,user_id,cta_clicked,copy_code_clicked,id,app_list,app_name,categories_l1,categories_l2,values
0,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,amazon prime video,amazon prime video,ott,rest,1
1,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
2,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,zomato,zomato,delivery_food,rest,1
3,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,alt balaji,alt balaji,ott,rest,1
4,5972f3e07f0aec4b796cb7e1,ctaClicked,copyCodeClicked,5972f3e07f0aec4b796cb7e1,bookmyshow,bookmyshow,entertainment,rest,1
...,...,...,...,...,...,...,...,...,...
2331347,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola,ola,travel_ridehailing,ride haling,1
2331348,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,ola driver app,ola driver app,driver_app,driver_app,1
2331349,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,rapido captain,rapido captain,driver_app,driver_app,1
2331350,63276f1c426a42baf332def0,ctaClicked,copyCodeClicked,63276f1c426a42baf332def0,snapchat,snapchat,social,rest,1


In [42]:
df6 = df_merge[~df_merge['copy_code_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['categories_l1'])\
                .agg(count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()

df6['total_customers'] = df_merge.user_id.nunique()
df6['%'] = (df6['count']*100.00/df6['total_customers']).round(2) 

In [43]:
df6.head(40)

,categories_l1,count,total_customers,%
0,social,97927,100770,97.18
1,tools,79217,100770,78.61
2,ott,78159,100770,77.56
3,ecommerce,72441,100770,71.89
4,finance_transactions,64521,100770,64.03
5,travel_ridehailing,61669,100770,61.20
6,streaming_music,60798,100770,60.33
7,delivery_food,54596,100770,54.18
8,office,45386,100770,45.04
9,finance_investment,28333,100770,28.12


In [44]:
category_list[category_list['categories_l1'] == 'finance_transactions']

,app_name,categories_l1,categories_l2
19,airtel thanks,finance_transactions,rest
42,axis mobile,finance_transactions,rest
84,cred,finance_transactions,rest
110,freecharge,finance_transactions,rest
131,bhim upi,finance_transactions,rest
132,google pay,finance_transactions,rest
201,mobikwik,finance_transactions,rest
239,paytm,finance_transactions,rest
241,paytm money,finance_transactions,rest
245,phonepe,finance_transactions,rest


In [47]:
# df_merge

In [58]:
df7 = df_merge[~df_merge['cta_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)]\
                .groupby(['categories_l1'])\
                .agg(apps  = ('app_name', set),
                     count = ('user_id', 'nunique'))\
                .sort_values(by='count', ascending=False)\
                .reset_index()
    
df7

,categories_l1,apps,count
0,social,"{linkedin, viber, mx takatak, sharechat, line,...",64093
1,ott,"{alt balaji, voot, amazon prime video, sonyliv...",51417
2,ecommerce,"{myntra, firstcry, meesho, nykaa, zivame, phar...",47854
3,finance_transactions,"{airtel thanks, freecharge, paytm money, mobik...",43547
4,travel_ridehailing,"{quick ride, in drive, blusmart, quickride, ub...",40441
5,streaming_music,"{wynk music, amazon music, spotify, hungama mu...",39672
6,delivery_food,"{swiggy, uber eats, zomato, dunzo}",35700
7,office,"{trello, zoho meeting, ms authenticator, onedr...",28726
8,tools,"{adobe scan, notion, file commander, google sh...",26799
9,finance_investment,"{kite by zerodha, nps, angel broking, namma pa...",18955


In [59]:
df7['total_customers'] = df_merge[~df_merge['cta_clicked'].isna()][~df_merge['app_list'].isin(most_common_apps)].user_id.nunique()
df7['%'] = (df7['count']*100.00/df7['total_customers']).round(2) 

In [60]:
df7

,categories_l1,apps,count,total_customers,%
0,social,"{linkedin, viber, mx takatak, sharechat, line,...",64093,65603,97.70
1,ott,"{alt balaji, voot, amazon prime video, sonyliv...",51417,65603,78.38
2,ecommerce,"{myntra, firstcry, meesho, nykaa, zivame, phar...",47854,65603,72.94
3,finance_transactions,"{airtel thanks, freecharge, paytm money, mobik...",43547,65603,66.38
4,travel_ridehailing,"{quick ride, in drive, blusmart, quickride, ub...",40441,65603,61.65
5,streaming_music,"{wynk music, amazon music, spotify, hungama mu...",39672,65603,60.47
6,delivery_food,"{swiggy, uber eats, zomato, dunzo}",35700,65603,54.42
7,office,"{trello, zoho meeting, ms authenticator, onedr...",28726,65603,43.79
8,tools,"{adobe scan, notion, file commander, google sh...",26799,65603,40.85
9,finance_investment,"{kite by zerodha, nps, angel broking, namma pa...",18955,65603,28.89
